# Exploring Crime Points

**Series:** Spatial Clustering (notebook 1 of 3)
**Time:** about 20 minutes

**In this notebook you will:**
- Load a year of Chicago crime reports
- Summarize crimes by type, month and hour
- Turn latitude and longitude into map points
- Reproject to meters, which clustering needs

**You should already know:** pandas basics and GeoDataFrames.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## About the data

The file holds about 200,000 crimes reported in Chicago during one year, from the
[Chicago Data Portal](https://data.cityofchicago.org/). Each row has a crime type, a date,
and a location.

In this series we ask: **where do certain crimes concentrate?** Clustering methods find
those concentrations automatically.

> Crime data describes reports, not all crime, and patterns reflect policing as well as behavior.
> Keep this in mind when interpreting any map.

## Step 1: Load the data

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Read the CSV straight from GitHub into memory (about 40 MB, nothing is saved to disk)
crimes = pd.read_csv("https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/CrimesChicago20220225.csv")
crimes.columns = crimes.columns.str.strip()          # some column names start with a space
crimes = crimes.dropna(subset=["LATITUDE", "LONGITUDE"])

In [ ]:
print(crimes.shape)
crimes[["DATE  OF OCCURRENCE", "PRIMARY DESCRIPTION", "LATITUDE", "LONGITUDE"]].head()

## Step 2: What types of crime are most common?

In [ ]:
crimes["PRIMARY DESCRIPTION"].value_counts().head(10).plot(kind="barh", figsize=(8, 4))
plt.gca().invert_yaxis()
plt.title("Ten most common crime types")
plt.xlabel("Number of reports")
plt.show()

## Step 3: When do crimes happen?

In [ ]:
crimes["date"] = pd.to_datetime(crimes["DATE  OF OCCURRENCE"], format="%m/%d/%Y %I:%M:%S %p")
crimes["hour"] = crimes["date"].dt.hour

crimes["hour"].value_counts().sort_index().plot(kind="bar", figsize=(10, 4))
plt.title("Crimes by hour of the day")
plt.xlabel("Hour (0 = midnight)")
plt.ylabel("Reports")
plt.show()

## Step 4: Choose three crime types

To keep things simple we focus on **weapons violations, robbery and narcotics**.

In [ ]:
crime_types = ["WEAPONS VIOLATION", "ROBBERY", "NARCOTICS"]
subset = crimes[crimes["PRIMARY DESCRIPTION"].isin(crime_types)]
subset["PRIMARY DESCRIPTION"].value_counts()

## Step 5: Make map points

In [ ]:
points = gpd.GeoDataFrame(
    subset,
    geometry=gpd.points_from_xy(subset["LONGITUDE"], subset["LATITUDE"]),
    crs="EPSG:4326",
)
points.plot(column="PRIMARY DESCRIPTION", markersize=0.5, legend=True, figsize=(7, 9))
plt.title("Three crime types in Chicago")
plt.axis("off")
plt.show()

## Step 6: Reproject to meters (important!)

Latitude and longitude are angles, not distances. One degree of longitude is about 85 km in Chicago
but one degree of latitude is about 111 km. Clustering methods measure distance, so we convert the
points to **UTM zone 16N** (`EPSG:32616`), where x and y are in **meters**.

In [ ]:
points = points.to_crs(epsg=32616)
points[["PRIMARY DESCRIPTION", "geometry"]].head()

The coordinates are now large numbers like `(448000, 4637000)`: meters east and north.

## Step 7: An interactive look at robberies

`explore()` draws on a web map. We sample 2,000 points so the map stays fast.

In [ ]:
points[points["PRIMARY DESCRIPTION"] == "ROBBERY"].sample(2000, random_state=1) \
    .explore(color="red", marker_kwds={"radius": 2}, tiles="Esri.WorldGrayCanvas")

## Try it yourself

Make a bar chart of **robberies by day of the week**. Which day has the most?

*Hint:* `crimes['date'].dt.day_name()` gives the weekday name.

<details>
<summary><b>Show a solution</b></summary>

```python
robberies = crimes[crimes["PRIMARY DESCRIPTION"] == "ROBBERY"]
robberies["date"].dt.day_name().value_counts().plot(kind="bar")
plt.title("Robberies by day of week")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- Always look at your data (counts, times, a quick map) before clustering.
- `points_from_xy` with `EPSG:4326` builds points from latitude and longitude.
- Reproject to a meter based system (here UTM 16N) before measuring distances.

**Next:** [DBSCAN and HDBSCAN](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/04b_DBSCAN_and_HDBSCAN.ipynb)